In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch import nn

In [15]:
SMALL_CAP = ["600169.SH", "000049.SZ", "600428.SH", "002064.SZ", "000501.SZ", "603000.SH", "000712.SZ", "600580.SH", "600017.SH", "603355.SH"]
MID_CAP = ["600138.SH", "000718.SZ", "002444.SZ", "600597.SH", "002285.SZ", "600729.SH", "000062.SZ", "002463.SZ", "002603.SZ", "603883.SH"]
LARGE_CAP = ["600718.SH", "600507.SH", "002583.SZ", "601718.SH", "600835.SH", "002268.SZ", "002092.SZ", "600098.SH", "002340.SZ", "300001.SZ"]

ASSETS = SMALL_CAP + MID_CAP + LARGE_CAP

In [16]:
DATA = Path("./data")
NUM_ASSETS = len(ASSETS)
WINDOW_SIZE = 10

# Prices and returns
daily_prices = pd.read_pickle(DATA / "daily_adj_close_post_20260630.pkl")
monthly_returns = pd.read_pickle(DATA / "monthly_returns.pkl")
prices = pd.read_hdf(DATA / "price.h5", key="price")

# Trading characteristics
daily_amount = pd.read_pickle(DATA / "daily_amount.pkl")
daily_turnover = pd.read_pickle(DATA / "daily_turnover.pkl")
daily_volume = pd.read_pickle(DATA / "daily_volume.pkl")

# Company characteristics
mcap = pd.read_pickle(DATA / "mcap.pkl")
industry = pd.read_pickle(DATA / "ind_code.pkl")

# Factor returns
ff5 = pd.read_csv(DATA / "FF5.csv", index_col=0, parse_dates=True)
hxz = pd.read_csv(DATA / "HXZ.csv", index_col=0, parse_dates=True)

# Risk-free rates
risk_free = pd.read_csv(DATA / "risk_free.csv")
shibor = pd.read_pickle(DATA / "shibor_rates.pkl")

# Stock selection and trading constraints
blacklist = pd.read_pickle(DATA / "BLACKLIST.pkl")
untradable = pd.read_pickle(DATA / "UNTRADABLE.pkl")
csi500_mask = pd.read_pickle(DATA / "csi500_mask_monthly.pkl")

In [17]:
FIRST_DATE = pd.Timestamp("2015-05-13")
FORMATION_DATE = pd.Timestamp("2018-12-31")

VAL_START = pd.Timestamp("2019-01-02")
VAL_END = pd.Timestamp("2020-12-31")

TEST_START = pd.Timestamp("2019-01-02")
TEST_END = LAST_DATE = pd.Timestamp("2024-08-30")

In [18]:
daily_prices = daily_prices.loc[:, ASSETS].copy()
daily_prices.index = pd.to_datetime(daily_prices.index)
daily_prices.index.name = "date"
daily_prices.columns.name = "asset"

daily_prices = daily_prices.sort_index().loc[FIRST_DATE:LAST_DATE]
daily_returns = daily_prices.pct_change(fill_method=None)

In [19]:
train_returns = daily_returns.loc[:FORMATION_DATE]
valid_returns = daily_returns.loc[VAL_START:VAL_END]
test_returns = daily_returns.loc[TEST_START:TEST_END]

for name, df in [
    ("Train", train_returns),
    ("Valid", valid_returns),
    ("Test", test_returns)
]:
    print(
        f"{name:12s} | "
        f"{df.index.min().date()} - {df.index.max().date()} | "
        f"{df.shape}"
    )

Train        | 2015-05-13 - 2018-12-28 | (890, 30)
Valid        | 2019-01-02 - 2020-12-31 | (487, 30)
Test         | 2019-01-02 - 2024-08-30 | (1376, 30)


In [20]:
class BaselineModel:
    def __init__(self, num_assets: int, window_size: int = 1, window_type: str = "rolling"):
        assert window_size > 0
        assert window_type in ("rolling", "expanding")

        self.num_assets = num_assets
        self.window_size = window_size
        self.window_type = window_type

    def forward(self, x: np.ndarray) -> np.ndarray:
        # x: (days, num_assets)
        # out: (days - window_size + 1, num_assets)
        assert x.ndim == 2
        assert x.shape[1] == self.num_assets
        assert x.shape[0] >= self.window_size

        out = []
        for i in range(x.shape[0] - self.window_size + 1):
            end = i + self.window_size
            inputs = x[i:end] if self.window_type == "rolling" else x[:end]
            out.append(np.mean(inputs, axis=0))

        return np.stack(out, axis=0)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)

In [21]:
model = BaselineModel(num_assets=NUM_ASSETS, window_size=WINDOW_SIZE, window_type="rolling")

# Training and Evaluation

In [22]:
# No training is required for the historical mean baseline

In [23]:
def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    assert y_true.shape == y_pred.shape
    assert y_true.ndim == 2
    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    errors = y_true - y_pred

    mse = np.mean(np.power(errors, 2))
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    directional_accuracy = np.mean(np.sign(y_true) == np.sign(y_pred))

    true_df = pd.DataFrame(y_true)
    pred_df = pd.DataFrame(y_pred)
    ic = pred_df.corrwith(true_df, axis=1, method="pearson")
    rank_ic = pred_df.corrwith(true_df, axis=1, method="spearman")

    return {
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "Directional Accuracy": directional_accuracy,
        "IC": ic.mean(),
        "Rank IC": rank_ic.mean()
    }

In [24]:
X_valid = valid_returns[:-1].to_numpy()
y_valid_true = valid_returns[WINDOW_SIZE:].to_numpy()
y_valid_pred = model(X_valid)
valid_metrics = evaluate(y_valid_true, y_valid_pred)
valid_metrics

{'MSE': np.float64(0.000739263594024058),
 'RMSE': np.float64(0.02718940223734347),
 'MAE': np.float64(0.019285672154840054),
 'Directional Accuracy': np.float64(0.47700908455625435),
 'IC': np.float64(-0.005182392135924183),
 'Rank IC': np.float64(-0.037212467323600236)}

# Out-Of-Sample

In [25]:
X_test = test_returns[:-1].to_numpy()
y_test_true = test_returns[model.window_size:].to_numpy()
y_test_pred = model(X_test)
test_metrics = evaluate(y_test_true, y_test_pred)
test_metrics

{'MSE': np.float64(0.0007240328425301796),
 'RMSE': np.float64(0.026907858378737236),
 'MAE': np.float64(0.01896192596371445),
 'Directional Accuracy': np.float64(0.47569546120058565),
 'IC': np.float64(0.001258639519560635),
 'Rank IC': np.float64(-0.025776447982129545)}

## Saving

In [26]:
test_predictions = pd.DataFrame(y_test_pred, index=test_returns.index[model.window_size:], columns=test_returns.columns)
test_predictions.to_csv("baseline_oos_predictions.csv")
display(test_predictions.head())
display(test_predictions.tail())

asset,600169.SH,000049.SZ,600428.SH,002064.SZ,000501.SZ,603000.SH,000712.SZ,600580.SH,600017.SH,603355.SH,...,600718.SH,600507.SH,002583.SZ,601718.SH,600835.SH,002268.SZ,002092.SZ,600098.SH,002340.SZ,300001.SZ
date,,,,,,,,,,,,,,,,,,,,,
2019-01-16,0.015172,0.000904,0.006905,0.003320,0.006024,0.011884,0.015618,0.008486,0.005406,0.006052,...,-0.001530,0.010290,0.016012,0.009525,0.010237,0.015277,0.002692,0.001803,0.004911,-0.003670
2019-01-17,0.012010,-0.000560,0.005756,0.003543,0.007941,0.011839,0.013774,0.007001,0.007237,0.003655,...,-0.000662,0.011782,0.013382,0.009496,0.007799,0.012407,0.002980,0.002159,0.004650,-0.003433
2019-01-18,0.009572,-0.003568,0.005124,0.000275,0.005418,0.009777,0.009273,0.005956,0.005436,0.002839,...,0.001755,0.010708,0.015348,0.005518,0.006850,0.009633,0.002422,0.002348,0.003906,-0.001720
2019-01-21,0.008163,-0.002621,0.003844,0.000273,0.004196,0.005675,0.000046,0.004910,0.003952,0.001815,...,-0.001282,0.013150,0.011516,0.002339,0.006154,0.008493,0.002653,0.001807,0.002549,-0.005267
2019-01-22,0.004312,-0.002544,0.003194,0.000039,0.003444,0.005376,-0.002902,0.000785,0.002139,0.002026,...,-0.001626,0.008061,0.006663,-0.001069,0.004792,0.005691,0.003314,0.001804,0.001280,-0.001033


asset,600169.SH,000049.SZ,600428.SH,002064.SZ,000501.SZ,603000.SH,000712.SZ,600580.SH,600017.SH,603355.SH,...,600718.SH,600507.SH,002583.SZ,601718.SH,600835.SH,002268.SZ,002092.SZ,600098.SH,002340.SZ,300001.SZ
date,,,,,,,,,,,,,,,,,,,,,
2024-08-26,-0.003761,-0.004064,-0.003372,-0.004184,-0.008458,-0.008362,0.005479,-0.009262,-0.001393,0.000521,...,-0.002021,-0.004096,-0.002960,-0.004446,-0.004255,-0.008154,0.000080,-0.002587,-0.002205,-0.006312
2024-08-27,-0.001572,0.001253,-0.000793,-0.002602,-0.006309,-0.006639,0.001189,-0.005663,-0.001044,-0.001045,...,-0.000718,-0.003327,-0.000405,-0.002807,-0.005022,-0.006677,0.003249,-0.002921,0.000264,-0.005163
2024-08-28,-0.003194,-0.005512,-0.000972,-0.001095,-0.005581,-0.008675,-0.007231,-0.010491,-0.001758,-0.002342,...,-0.002032,-0.004097,-0.004286,-0.004433,-0.006865,-0.008046,0.000950,-0.004387,-0.002155,-0.010736
2024-08-29,-0.002112,-0.003528,0.001203,0.001628,-0.001172,-0.006059,-0.013007,-0.008110,-0.001043,-0.002157,...,-0.000583,-0.002824,-0.002987,-0.002801,-0.007010,-0.006529,0.001525,-0.004387,-0.000257,-0.008769
2024-08-30,-0.002089,0.001019,-0.000098,0.001046,-0.004759,-0.007684,-0.014710,-0.004298,-0.004629,-0.000753,...,-0.001100,-0.002824,-0.002648,-0.002790,-0.004524,-0.005182,0.001518,-0.005036,0.000271,-0.005789
